In [13]:
!pip install -q -U diffusers transformers accelerate peft
!pip install -q PyMuPDF patool spandrel easyocr controlnet-aux safetensors ftfy regex tqdm

print("\n\n✅ DONE! Please Restart the Kernel.")



✅ DONE! Please Restart the Kernel.


In [ ]:
"""
Comic Book Auto-Colorizer
===============================================================
"""

import hashlib
import os, gc, sys, json, subprocess, shutil, glob, zipfile, traceback, tempfile

os.environ["TF_CPP_MIN_LOG_LEVEL"]  = "3"
os.environ["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"

import patoolib, fitz, cv2, numpy as np, torch
from PIL import Image, ImageDraw
from tqdm import tqdm

# ===========================================================
# CONFIGURATION
# ===========================================================
INPUT_FOLDER  = "/kaggle/input" 
OUTPUT_FOLDER = "/kaggle/working/output"

TEMP_IN    = "/kaggle/working/temp_extract"
TEMP_COLOR = "/kaggle/working/temp_colored"
TEMP_OUT   = "/kaggle/working/temp_upscaled"

INFERENCE_WIDTH  = 512   
COMPOSITE_WIDTH  = 1024  

CONTROLNET_SCALE = 0.95
IP_ADAPTER_SCALE = 0.25  
NUM_STEPS        = 25
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

OCR_BOX_PAD = 10   

ESRGAN_URL  = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
ESRGAN_PATH = "/kaggle/working/RealESRGAN_x4plus.pth"

# ===========================================================
# UTILITIES
# ===========================================================

def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def clean_temps():
    for d in [TEMP_IN, TEMP_COLOR, TEMP_OUT]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

def extract_pdf(path, out):
    doc = fitz.open(path)
    for i in range(len(doc)):
        doc.load_page(i).get_pixmap(dpi=300).save(
            os.path.join(out, f"page_{i:04d}.jpg"))
    doc.close()

def list_images(folder):
    imgs = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                imgs.append(os.path.join(root, f))
    return sorted(imgs)

def comic_seed(name):
    return int(hashlib.md5(name.encode()).hexdigest()[:8], 16) % (2 ** 32)

# ===========================================================
# STAGE 1 — OCR (Aggressive Settings)
# ===========================================================

_OCR_SCRIPT = r"""
import sys, json, cv2, numpy as np
import easyocr

PAD = int(sys.argv[3]) if len(sys.argv) > 3 else 10
reader = easyocr.Reader(['en'], gpu=False, verbose=False)
image_paths = json.loads(sys.argv[1])
out_file    = sys.argv[2]
results     = {}

for p in image_paths:
    img = cv2.imread(p)
    if img is None:
        results[p] = []
        continue
        
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h_img, w_img = img.shape[:2]
    
    try:
        h_boxes, _ = reader.detect(gray_img, text_threshold=0.2, low_text=0.2, link_threshold=0.2)
        boxes = []
        if h_boxes and len(h_boxes[0]) > 0:
            for box in h_boxes[0]:
                x_min, x_max, y_min, y_max = [int(v) for v in box]
                x_min = max(0,     x_min - PAD)
                y_min = max(0,     y_min - PAD)
                x_max = min(w_img, x_max + PAD)
                y_max = min(h_img, y_max + PAD)
                boxes.append([x_min, y_min, x_max, y_max])
        results[p] = boxes
    except Exception:
        results[p] = []

with open(out_file, "w") as f:
    json.dump(results, f)
"""

def run_ocr(image_paths):
    print(f"  [OCR] Processing {len(image_paths)} pages (CPU, subprocess)...")
    worker = "/kaggle/working/ocr_worker.py"
    result = "/kaggle/working/ocr_results.json"
    with open(worker, "w") as f:
        f.write(_OCR_SCRIPT)

    try:
        subprocess.run(
            [sys.executable, worker, json.dumps(image_paths), result, str(OCR_BOX_PAD)],
            capture_output=True, text=True, timeout=1200)
    except subprocess.TimeoutExpired:
        return {}

    try:
        with open(result) as f:
            return json.load(f)
    except Exception:
        return {}

def erase_text(image_paths, ocr_results):
    orig, erased = {}, {}
    for p in image_paths:
        img = cv2.imread(p)
        if img is None: continue
        
        img = cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
        e = img.copy()
        
        for box in ocr_results.get(p, []):
            x1, y1, x2, y2 = box
            cv2.rectangle(e, (x1, y1), (x2, y2), (255, 255, 255), -1)
        orig[p], erased[p] = img, e
    return orig, erased

# ===========================================================
# STAGE 2 — SD 1.5 + CONTROLNET + IP-ADAPTER
# ===========================================================

def load_pipeline():
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
    from controlnet_aux import LineartDetector
    print("  [SD] Loading LineartDetector...")
    detector = LineartDetector.from_pretrained("lllyasviel/Annotators")

    print("  [SD] Loading ControlNet...")
    cn = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_lineart",
        torch_dtype=torch.float16)

    print("  [SD] Loading SD 1.5 & IP-Adapter...")
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=cn,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False)

    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
    pipe.set_ip_adapter_scale(IP_ADAPTER_SCALE)

    pipe = pipe.to("cpu")
    return pipe, detector

def pipe_to_gpu(pipe):
    pipe.unet         = pipe.unet.to(DEVICE)
    pipe.vae          = pipe.vae.to(DEVICE)
    pipe.text_encoder = pipe.text_encoder.to(DEVICE)
    pipe.controlnet   = pipe.controlnet.to(DEVICE)
    try: pipe.image_encoder = pipe.image_encoder.to(DEVICE)
    except: pass
    
    # CRITICAL BUG FIX: Removed xformers and attention_slicing so they 
    # don't overwrite the IP-Adapter processors!

def pipe_to_cpu(pipe):
    pipe.unet         = pipe.unet.to("cpu")
    pipe.vae          = pipe.vae.to("cpu")
    pipe.text_encoder = pipe.text_encoder.to("cpu")
    pipe.controlnet   = pipe.controlnet.to("cpu")
    try: pipe.image_encoder = pipe.image_encoder.to("cpu")
    except: pass
    flush()

def make_lineart(img_cv, detector):
    h, w = img_cv.shape[:2]
    th   = (int(INFERENCE_WIDTH * h / w) // 8) * 8
    pil  = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)).resize((INFERENCE_WIDTH, th), Image.LANCZOS)
    return detector(pil, detect_resolution=INFERENCE_WIDTH, image_resolution=INFERENCE_WIDTH)

def colorize(lineart_pil, cover_pil, pipe, seed):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    out = pipe(
        prompt=(
            "comic book page, professional modern comic coloring, "
            "natural warm human skin tones, vibrant jungle background, lush green foliage, "
            "blue water, cool water tones, dynamic cel shading, crisp sharp inks, "
            "high contrast impact bursts, bold action panel colors, "
            "bright yellow and white speed lines, vibrant action effects"
        ),
        negative_prompt=(
            "monochrome, sepia, green skin, teal skin, neon, glowing, "
            "text, letters, speech bubbles, blurry, "
            "muted action effects, brown speed lines, dull impact colors"
        ),
        image=lineart_pil,
        ip_adapter_image=cover_pil, 
        num_inference_steps=NUM_STEPS,
        controlnet_conditioning_scale=CONTROLNET_SCALE,
        guidance_scale=7.0,
        generator=generator,
    ).images[0]
    flush()
    return out

# ===========================================================
# SMART COMPOSITING
# ===========================================================

def composite_before_upscale(color_pil, orig_cv, ocr_boxes):
    orig_h, orig_w = orig_cv.shape[:2]
    comp_h = int(COMPOSITE_WIDTH * orig_h / orig_w)
    scale = COMPOSITE_WIDTH / orig_w 

    color_scaled = color_pil.resize((COMPOSITE_WIDTH, comp_h), Image.LANCZOS)
    color_cv = cv2.cvtColor(np.array(color_scaled), cv2.COLOR_RGB2BGR)

    orig_scaled = cv2.resize(orig_cv, (COMPOSITE_WIDTH, comp_h), interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(orig_scaled, cv2.COLOR_BGR2GRAY)

    if ocr_boxes:
        for box in ocr_boxes:
            x1, y1, x2, y2 = [int(v * scale) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(COMPOSITE_WIDTH, x2), min(comp_h, y2)
            
            if x2 > x1 and y2 > y1:
                roi_gray = gray[y1:y2, x1:x2]
                roi_color = color_cv[y1:y2, x1:x2]
                
                white_mask = roi_gray > 220
                roi_color[white_mask] = (255, 255, 255)
                color_cv[y1:y2, x1:x2] = roi_color

    gray_clean = np.clip((gray.astype(np.float32) - 60) * (255.0 / (200 - 60)), 0, 255)
    final_cv = (color_cv.astype(np.float32) * (gray_clean[:, :, None] / 255.0)).astype(np.uint8)

    # ADD THIS: force near-white original pixels back to pure white
    # Protects panel gutters and speech bubble backgrounds
    white_protect = gray > 250
    final_cv[white_protect] = (255, 255, 255)

    return Image.fromarray(cv2.cvtColor(final_cv, cv2.COLOR_BGR2RGB))

# ===========================================================
# STAGE 3 — 4x UPSCALE
# ===========================================================

def load_upscaler():
    import urllib.request, spandrel
    if not os.path.exists(ESRGAN_PATH):
        urllib.request.urlretrieve(ESRGAN_URL, ESRGAN_PATH)
    return spandrel.ModelLoader().load_from_file(ESRGAN_PATH).eval().to(DEVICE)

def unload_upscaler(m):
    del m; flush()

def upscale(src, dst, model, tile=512, pad=16):
    bgr = cv2.imread(src, cv2.IMREAD_COLOR)
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    t    = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).float() / 255.0
    sc   = 4
    out  = torch.zeros(1, 3, h*sc, w*sc)
    with torch.no_grad():
        for y in range(0, h, tile):
            for x in range(0, w, tile):
                x1,y1 = max(0,x-pad),      max(0,y-pad)
                x2,y2 = min(w,x+tile+pad),  min(h,y+tile+pad)
                p  = t[:,:,y1:y2,x1:x2].to(DEVICE)
                op = model(p).cpu()
                ox1 = (x-x1)*sc;  oy1 = (y-y1)*sc
                ox2 = ox1 + min(tile,w-x)*sc
                oy2 = oy1 + min(tile,h-y)*sc
                dx1,dy1 = x*sc, y*sc
                dx2 = min(w*sc, dx1+min(tile,w-x)*sc)
                dy2 = min(h*sc, dy1+min(tile,h-y)*sc)
                out[:,:,dy1:dy2,dx1:dx2] = op[:,:,oy1:oy2,ox1:ox2]
    res = (out.squeeze(0).permute(1,2,0).numpy()*255).clip(0,255).astype(np.uint8)
    cv2.imwrite(dst, cv2.cvtColor(res, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])

# ===========================================================
# MAIN
# ===========================================================

def get_cover(archive_path):
    tmp = tempfile.mkdtemp()
    try:
        patoolib.extract_archive(archive_path, outdir=tmp, interactive=False)
        imgs = list_images(tmp)
        if imgs:
            return Image.open(imgs[0]).convert("RGB")
    except Exception:
        pass
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    return Image.new("RGB", (512, 512), (200, 200, 200))

def process_comic(path, idx, pipe, detector, cover_pil):
    name      = os.path.basename(path)
    base_name = os.path.splitext(name)[0]
    outpath   = os.path.join(OUTPUT_FOLDER, f"{base_name}_Colorized.cbz")

    counter = 1
    while os.path.exists(outpath):
        outpath = os.path.join(OUTPUT_FOLDER, f"{base_name}_Colorized_{counter}.cbz")
        counter += 1

    seed = comic_seed(name)
    clean_temps()

    lo = path.lower()
    if   lo.endswith(".pdf"): extract_pdf(path, TEMP_IN)
    elif lo.endswith((".cbz",".zip",".cbr",".rar")): patoolib.extract_archive(path, outdir=TEMP_IN, interactive=False)
    else: return

    images = list_images(TEMP_IN)
    if not images: return

    ocr_data = run_ocr(images)
    orig_cache, erased_cache = erase_text(images, ocr_data)

    pipe_to_gpu(pipe)

    with tqdm(total=len(images), desc=f"{name[:28]:<28}", position=idx, leave=True) as bar:
        for img_path in images:
            base      = os.path.basename(img_path)
            color_dst = os.path.join(TEMP_COLOR, base)
            orig_cv   = orig_cache.get(img_path)
            erased_cv = erased_cache.get(img_path)
            
            page_boxes = ocr_data.get(img_path, [])

            if orig_cv is not None:
                try:
                    la  = make_lineart(erased_cv, detector)
                    col = colorize(la, cover_pil, pipe, seed)
                    
                    col = composite_before_upscale(col, orig_cv, page_boxes)
                    col.save(color_dst, quality=95)
                except Exception as e:
                    traceback.print_exc()
                    shutil.copy(img_path, color_dst)
            else:
                shutil.copy(img_path, color_dst)
            bar.update(1)

    pipe_to_cpu(pipe)

    sr = load_upscaler()
    for img_path in images:
        base = os.path.basename(img_path)
        src  = os.path.join(TEMP_COLOR, base)
        dst  = os.path.join(TEMP_OUT,   base)
        src  = src if os.path.exists(src) else img_path
        try: upscale(src, dst, sr)
        except Exception: shutil.copy(src, dst)
    unload_upscaler(sr)

    with zipfile.ZipFile(outpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(TEMP_OUT)):
            z.write(os.path.join(TEMP_OUT, f), arcname=f)

def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    files = []
    for pat in ("*.cbz","*.cbr","*.zip","*.rar","*.pdf"):
        files.extend(glob.glob(os.path.join(INPUT_FOLDER, "**", pat), recursive=True))
    if not files: return

    print("[INIT] Loading SD pipeline on CPU...")
    pipe, detector = load_pipeline()

    ref_images = []
    for pat in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"):
        ref_images.extend(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))
        
    if ref_images:
        ref_images.sort(key=lambda x: "ref" in x.lower(), reverse=True)
        print(f"[INIT] Found direct reference image: {os.path.basename(ref_images[0])}")
        cover_pil = Image.open(ref_images[0]).convert("RGB")
    else:
        print("[INIT] No loose image found. Preparing reference from first comic cover...")
        cover_pil = get_cover(files[0])

    for i, fp in enumerate(files):
        print(f"\n[{i+1}/{len(files)}] {os.path.basename(fp)}")
        try: process_comic(fp, i, pipe, detector, cover_pil)
        except Exception: traceback.print_exc()

    del pipe, detector; flush()
    clean_temps()
    print("\nAll done!")

if __name__ == "__main__":
    main()

[INIT] Loading SD pipeline on CPU...
  [SD] Loading LineartDetector...
  [SD] Loading ControlNet...
  [SD] Loading SD 1.5 & IP-Adapter...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: h94/IP-Adapter
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` 

[INIT] Found direct reference image: ref.png

[1/1] Jungle-Fury.cbz


INFO patool: ... /kaggle/input/datasets/wildscop/jungle-fury/Jungle-Fury.cbz extracted to `/kaggle/working/temp_extract'.


  [OCR] Processing 5 pages (CPU, subprocess)...
